# 0922 17일차

## 1. 증폭으로 훈련셋 늘리기

원본에서 일부를 뽑아 변형한 뒤 원본에 이어붙여 훈련 데이터를 늘리는 방법

**필요한 이유**

1. 데이터가 적으면 모델이 훈련 데이터를 외우게 됨
2. 사진을 더 찍어올 수 없는 데이터라도 변형으로 늘릴 수 있음
3. 원본 파일은 그대로 두고 메모리에서만 만들어 냄

- 변형 옵션은 생성기를 만들 때 지정하고, 실제로 만드는 것은 `flow`
- 과적합일 때 쓰는 수단이므로, 과적합이 아닌데 먼저 넣으면 학습만 어려워짐

### 1-1. 절차

1. 몇 장을 더 만들지 정함
2. 원본에서 그만큼 **번호를 무작위로 뽑음**
3. 그 번호의 사진과 라벨을 꺼냄
4. 생성기에 넣고 배치를 통째로 꺼냄
5. 원본 뒤에 이어붙임

![사진 10장에서 5장을 뽑아 변형한 뒤 원본에 이어붙이는 과정. 번호표를 뽑아 그 번호의 사진을 꺼내고, 변형 기계를 통과시킨 뒤 원본 뒤에 붙여 15장이 된다](assets/augment-steps.svg)

- 2번에서 나오는 것은 **사진이 아니라 번호**임. 그 번호로 사진과 라벨을 같이 꺼내야 짝이 맞음
- 3번까지는 원본 그대로임. 변형은 생성기를 통과할 때 일어남
- 생성기에 **넣기 전의 배열과 꺼낸 배열은 다른 것**. 꺼낸 쪽을 붙여야 증폭이 들어감
- 같은 번호가 두 번 뽑혀도 변형이 다르게 들어가므로 서로 다른 사진이 됨
- 원본을 대체하는 것이 아니라 추가함. 원본과 변형본이 둘 다 있어야 같은 것이 조금 다르게 생긴 경우를 배움
- 라벨은 변형되지 않음. 셔츠를 5도 돌려도 셔츠임
- 생성기가 스케일까지 조정하므로, 붙일 원본도 같은 범위로 맞춰야 함

### 1-2. 무작위로 뽑기

번호를 무작위로 뽑는 두 가지 방법이 있고, **중복을 끌 수 있느냐**가 다름

1. `randint` : 중복을 끌 수 없음. 위쪽 끝은 포함하지 않으므로 전체 장수를 그대로 넣어도 됨
2. `choice` : 기본은 중복 허용이고 끌 수 있음. 끄면 전체 개수보다 많이 뽑지 못함

- 증폭에서는 중복이 문제되지 않음. 같은 장이 여러 번 뽑혀도 변형이 매번 다르게 들어가므로 서로 다른 이미지가 됨
- 4만 개에서 2만 5천 개를 뽑으면 실제로 서로 다른 것은 1만 8천 개 정도

### 1-3. 증폭 옵션 고르기

데이터에 **실제로 있을 법한 변형**만 씀

1. 좌우 반전 : 대부분의 사물에 자연스러움
2. 위아래 반전 : 뒤집힌 자동차나 사람은 현실에 없음. 위성사진처럼 방향이 없는 데이터에만
3. 회전·기울이기 : 단위가 **각도**임. 비율로 착각해 작은 값을 주면 변형이 거의 일어나지 않음
4. 평행이동·확대 : 단위가 비율임

- 없는 상황을 학습시키면 오히려 정확도가 떨어짐
- 단위를 착각하면 에러가 나지 않으므로, 만들어진 이미지를 그려서 확인해야 알 수 있음

#### 주의) 검증셋은 증폭 전에 나눔

훈련할 때 쓰는 **검증 비율 지정은 섞기 전 데이터의 뒷부분**을 떼어감

1. 증폭본을 뒤에 이어붙였으므로 검증셋이 전부 증폭본이 됨
2. 증폭본은 훈련셋에 있는 원본을 변형한 것이라, 이미 본 이미지로 검증하는 셈
3. 테스트는 변형 없는 원본인데 검증은 변형된 이미지라, 서로 다른 것을 재게 됨

- 조기 종료와 체크포인트가 이 검증 손실을 보고 판단하므로 기준 자체가 어긋남
- 증폭 전에 원본에서 검증셋을 떼어 두고, 훈련셋만 증폭해서 검증 데이터를 직접 넘김
- 증폭은 훈련 데이터에만 적용한다는 원칙과도 맞음

## 2. 증폭 코드에 쓰인 문법

절차는 다섯 동작인데 코드가 짧아 보이는 이유는, 넘파이가 여러 개를 한 번에 처리하는 문법을 쓰기 때문

### 2-1. 팬시 인덱싱

대괄호에 숫자 하나가 아니라 **번호 목록**을 넣으면, 그 개수만큼 한 번에 꺼냄

```python
x[3]                    # 1개
x[[3, 7, 3, 0, 8]]      # 5개. 목록 순서 그대로
```

```python
randidx = np.random.randint(x_train.shape[0], size=augment_size)
x_augmented = x_train[randidx]      # 그 번호들의 사진
y_augmented = y_train[randidx]      # 같은 번호의 정답
```

1. 목록에 적힌 순서대로 나옴
2. 같은 번호가 두 번 있으면 그 사진도 두 번 나옴
3. 하나씩 꺼내 쌓는 것과 결과가 같고, 한 줄로 끝남

- x와 y에 **같은 목록**을 넣어야 사진과 정답의 순서가 어긋나지 않음. 따로 뽑으면 3번 사진에 7번 정답이 붙을 수 있음
- `shape[0]`은 장수. 뽑을 번호의 범위를 여기에 맞춰야 없는 번호를 부르지 않음
- `size`는 몇 개 뽑을지

### 2-2. 메서드 체이닝

점을 연달아 찍어 **두 줄을 한 줄로** 붙여 쓰는 방식

```python
기계 = data_gen.flow(x_augmented, y_augmented, batch_size=augment_size)
xy_augmented = 기계.next()
```

```python
xy_augmented = data_gen.flow(x_augmented, y_augmented, batch_size=augment_size).next()
```

1. 앞의 결과에 바로 다음 동작을 이어 붙임
2. 중간 변수를 만들지 않아도 됨

- 한 줄로 보이지만 **두 가지 일**이 일어남. `flow`는 기계를 만들 뿐이고 꺼내는 것은 `next`
- `next`를 빼면 기계만 만들고 결과를 안 꺼낸 상태가 됨

### 2-3. 묶음으로 받는 인자

이어붙이는 함수는 재료를 **하나의 묶음으로** 받으므로 괄호가 두 겹이 됨

```python
np.concatenate((x_train, xy_augmented[0]))
#              ↑                        ↑
#            바깥은 함수 호출 / 안쪽은 이어붙일 것들의 묶음
```

1. 안쪽 묶음에 셋 이상을 넣어도 됨
2. 묶음 없이 나열하면 두 번째 자리를 다른 뜻(어느 축으로 붙일지)으로 읽어 에러가 남

- 괄호가 중복돼 보이지만 역할이 다름

### 2-4. copy

꺼낸 배열이 원본과 메모리를 공유할 수 있으므로, 따로 떼어 두는 것

```python
x_augmented = x_train[randidx].copy()
```

1. 공유된 상태에서 한쪽을 바꾸면 다른 쪽도 같이 바뀜
2. 증폭본을 건드릴 때 원본이 변하는 것을 막음

- 지금 절차에서는 없어도 문제가 드러나지 않지만, 붙여 두면 확실함

## 3. 데이터셋마다 다른 형태

케라스 기본 데이터셋은 흑백 계열과 컬러 계열의 형태가 달라서, 같은 코드를 옮길 때 걸림

| | x | y |
|---|---|---|
| 흑백 (mnist, fashion) | `(장수, 세로, 가로)` 3차원 | `(장수,)` 1차원 |
| 컬러 (cifar10, cifar100) | `(장수, 세로, 가로, 채널)` 4차원 | `(장수, 1)` 2차원 |

- 모델과 생성기는 언제나 x를 4차원으로 받음

### 3-1. 채널 축 붙이기

흑백은 채널 축이 생략돼 있으므로 붙여야 하고, 컬러는 이미 4차원이라 그대로 씀

1. 흑백 : 뒤에 `1`을 붙여 4차원으로 만듦
2. 컬러 : 손댈 것 없음

- 흑백 코드를 컬러에 그대로 옮기면 채널 축을 또 붙이게 되어 값의 개수가 맞지 않음
- 형태를 바꾸는 것은 값의 개수를 그대로 두고 칸 나누는 방법만 바꾸는 일이므로, 앞뒤의 곱이 같아야 함

### 3-2. 원핫 인코딩 위치

y가 번호로 오는 데이터셋은 직접 원핫으로 바꿔야 함

1. 흑백 계열은 y가 1차원이라, 인코더가 2차원만 받으므로 형태를 먼저 바꿈
2. 컬러 계열은 y가 이미 2차원이라 그대로 넣으면 됨

- **증폭본을 붙인 뒤에 한 번만** 변환함. 붙이기 전에 하면 한쪽은 원핫이고 한쪽은 번호라 이어붙일 수 없음
- 검증셋과 테스트셋은 인코더를 **다시 학습시키지 말고 변환만** 함. 다시 학습하면 그 안에 없는 클래스만큼 열이 줄어 모델 출력과 맞지 않게 됨
- 클래스 분포를 확인하려면 원핫으로 바꾸기 전에 세야 함. 원핫 뒤에는 0과 1의 개수만 나옴

#### 주의) BatchNormalization과 Dropout 순서

`BatchNormalization`은 들어온 값들의 평균과 분산을 계산해 기억해 두었다가 예측할 때 씀

1. 앞에 `Dropout`이 있으면 일부가 **꺼진 상태의 통계**를 배움
2. 예측할 때는 `Dropout`이 꺼지므로 값의 분포가 달라짐
3. 기억해 둔 통계와 실제가 맞지 않게 됨

- `BatchNormalization`을 먼저 두고 `Dropout`을 뒤에 둠
- 일반적인 순서는 합성곱 → 정규화 → 활성화 → `Dropout`

## 4. 경사하강법

loss가 가장 낮아지는 w를 찾아 조금씩 내려가는 방법. 미분으로 방향을 정하고 lr만큼 움직임

### 4-1. w와 loss의 관계

`w`를 바꾸면 loss가 달라지고, 그 관계를 그리면 통상 **U자(2차함수)** 모양

1. 바닥인 **꼭짓점**이 최적의 `w`
2. 그 최적값을 모르므로, 지금 `w`가 얼마나 나쁜지를 loss로 잼

- 실제 신경망은 가중치가 수십만 개라 U자가 아니라 울퉁불퉁한 고차원 지형. 개념을 잡을 때만 단순화해 그림
- **주의)** 꼭짓점과 변곡점은 다름. 변곡점은 휘는 방향이 바뀌는 지점이고, 2차함수는 처음부터 끝까지 아래로 볼록이라 변곡점이 없음

### 4-2. 갱신식

```
w = w - lr · ∂L/∂w
```

예측이 `ŷ = wx`, 손실이 `L = Σ(y - ŷ)²`일 때

```
∂L/∂w = Σ 2(y - wx) · (-x) = -2 Σ x(y - wx)

w = w + 2·lr · Σ x(y - wx)
```

1. 안쪽의 `(y - wx)`를 `w`로 미분하면 `-x`가 나오고, 이것이 곱해짐 (연쇄법칙)
2. 그 마이너스와 경사하강법의 마이너스가 상쇄되어 부호가 `+`가 됨
3. `2`는 제곱을 미분해서 나온 계수. `lr`에 흡수시켜 생략하기도 함

- 미분값은 그 지점의 경사. 어느 쪽으로 가야 loss가 주는지(방향)는 알려주지만, 최저점까지 얼마나 먼지(거리)는 알려주지 않음
- 그래서 한 번에 못 가고 **조금씩 여러 번** 움직임

### 4-3. learning_rate

한 걸음의 **보폭**

1. 작으면 : 한 방향으로 천천히 내려감. 성능 향상이 너무 느림
2. 크면 : 초반은 빠르지만 바닥 근처에서 넘나들기만 해서 더 줄이지 못함
3. 너무 크면 : 튕겨나가 loss가 오히려 커짐

- 바닥에 가까워지면 기울기가 작아져 보폭도 줄지만, `lr`이 고정이라 충분히 줄지 않아 진동함
- 큰 `lr`로 시작해 바닥 근처에서 줄이는 방법이 있음. `ReduceLROnPlateau`가 `val_loss`가 나아지지 않으면 `lr`을 줄여줌
- 이 진동은 훈련 후반, `val_loss`가 올라가기 시작하는 **과적합 구간과 시점이 겹침**
  - 원인은 다름. 하나는 보폭이 충분히 안 줄어서이고, 하나는 훈련 데이터를 외워서임
  - 훈련 데이터를 거의 다 맞히게 된 시점이 곧 외우기 시작하는 시점이라 같이 나타남
  - `ReduceLROnPlateau`와 `EarlyStopping`이 비슷한 시점에 동작하는 이유. 둘 다 `val_loss`를 보고 움직임

### 4-4. 골짜기가 여러 개일 때

loss 지형에 골짜기가 둘 이상이면, **출발 위치에 따라 다른 골짜기에 도착함**

1. 기울기는 발밑 경사만 알려줌
2. 골짜기 바닥에 오면 기울기가 0이라 멈춤
3. 건너편에 더 깊은 골짜기가 있는지는 알 방법이 없음

- 어느 골짜기가 가장 깊은지 알 수 없으므로, `random_state`·`lr`·`optimizer`·구조를 바꿔가며 여러 번 돌려 나은 쪽을 고름. 튜닝이 필요한 이유
- `random_state`를 고정하는 이유도 여기 있음. 고정하지 않으면 같은 코드인데 결과가 달라져 무엇 때문에 좋아졌는지 알 수 없음
- 가중치가 수십만 개면 내려갈 길이 하나라도 있을 확률이 높아, 실무에서 지역 최저점 자체는 크게 걱정하지 않음

### 4-5. 역전파를 손으로 쓰지 않는 이유

가중치를 고치는 식을 **사람이 쓰지 않는다**는 뜻이지, 역전파를 안 한다는 뜻이 아님

1. 층이 쌓이면 연쇄법칙이 층마다 이어져 식이 감당이 안 됨
2. RNN은 같은 가중치를 시간 방향으로 여러 번 재사용하므로 그만큼 더 거슬러 올라가야 함
   - 시간을 따라가는 역전파를 BPTT라고 함

- 프레임워크가 계산 과정을 기록해 두었다가 거꾸로 되짚으며 연쇄법칙을 적용함. 자동 미분이라고 함
- 모델은 여전히 역전파로 학습함. 달라지는 것은 사람이 수식을 전개하느냐임

### 4-6. 사람이 고르는 두 가지

미분과 갱신이 자동이므로, 학습 방식에서 사람이 고르는 것은 둘뿐

1. `loss` : 무엇을 줄일지
2. `optimizer` : 어떻게 줄일지. `learning_rate`가 여기 포함됨

- `compile`이 그 선언이고, `fit`을 부르면 미분과 갱신은 안에서 일어남

### 4-7. 영향이 큰 순서

학습 방식보다 **데이터 쪽이 훨씬 크게 좌우함**

1. 데이터의 양과 질 : 클래스 균형, 실제 데이터와 얼마나 닮았는지
2. 데이터와 모델의 짝 : 어긋나면 성능 이전에 학습 자체가 안 됨
3. 구조 : 깊이와 filters
4. 규제 : 과적합일 때의 Dropout과 증폭
5. 학습 설정 : 4-6에서 고르는 것들

- 데이터가 기울어 있으면 한쪽으로만 찍어도 그 비율만큼 나오므로, 정확도만 보고는 학습 여부를 알 수 없음
- 데이터가 쉬우면 구조가 어설퍼도 맞으므로, 구조를 비교하려면 어려운 데이터로 해야 함
- 구조를 아무리 손봐도 데이터가 부족하면 한계가 있음. 증폭이 필요한 이유
- 5번은 기본값으로 두고 거의 건드리지 않음. 직접 고르는 자리인데도 영향이 가장 작음